# Evaluación — Delivery Promise

Este notebook resume métricas del holdout temporal y el tradeoff cobertura vs ancho de intervalo.

Requisito: haber corrido `python scripts/run_pipeline.py`.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

metrics = json.loads((ROOT / "artifacts" / "metrics.json").read_text(encoding="utf-8"))
pd.Series(metrics)

In [ ]:
from src.data.generate_synthetic import generate_orders
from src.models.train import train_and_persist
from src.models.evaluate import evaluate_promise
import numpy as np

# Re-entrenar en memoria para explorar sensibilidad del cuantil alto
orders = generate_orders(n_orders=8000, seed=42)
result = train_and_persist(orders, ROOT / "artifacts")
y = result["test_df"]["total_delivery_minutes"].to_numpy()
p20, p50, p80 = result["preds"][0.2], result["preds"][0.5], result["preds"][0.8]

rows = []
for scale in [0.9, 1.0, 1.1, 1.2]:
    m = evaluate_promise(y, p20, p50, p50 + (p80 - p50) * scale)
    m["high_scale"] = scale
    rows.append(m)
tradeoff = pd.DataFrame(rows)
tradeoff

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(tradeoff["mean_width_minutes"], tradeoff["coverage"], marker="o")
for _, r in tradeoff.iterrows():
    ax.annotate(f"x{r['high_scale']}", (r["mean_width_minutes"], r["coverage"]))
ax.set_xlabel("Ancho medio del intervalo (min)")
ax.set_ylabel("Coverage")
ax.set_title("Tradeoff competitividad vs cobertura")
ax.grid(True, alpha=0.3)
plt.show()